# Cybersecurity — задания C1.1–C1.6 и C2.1–C2.6

12 практических заданий на Python.

Примеры основаны на искусственных учебных данных; реальные серверы и сети не используются.

## Подготовка

In [ ]:
from pathlib import Path
for name in ["seclab", "seclab/c1", "seclab/c2", "examples", "examples/data", "tests"]:
    folder = Path(name)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "__init__.py").touch()
print("Created the source, example and test directories.")


## C1 — Исключения, даты, регулярные выражения и файлы

### C1.1 — Разбор SSH-логов

In [ ]:
%%writefile seclab/c1/auth.py
from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import TypedDict

log = logging.getLogger(__name__)

class AuthEvent(TypedDict):
    timestamp: str
    host: str
    result: str
    user: str
    invalid_user: bool
    ip: str
    port: int

class MalformedLogLine(ValueError):
    pass

SSH_PATTERN = re.compile(
    r"(?P<timestamp>[A-Z][a-z]{2}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+"
    r"(?P<host>[\w.-]+)\s+sshd\[\d+\]:\s+"
    r"(?P<result>Failed|Accepted)\s+"
    r"(?:password|publickey|keyboard-interactive(?:/pam)?)\s+for\s+"
    r"(?P<invalid>invalid\s+user\s+)?"
    r"(?P<user>\S+)\s+from\s+(?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+"
    r"port\s+(?P<port>\d+)\s+ssh2"
)

class AuthLogFileError(OSError):
    pass

def parse_ssh_line(line: str) -> AuthEvent | None:
    text = line.strip()
    if not text:
        return None
    if "sshd" not in text and not text.startswith("### corrupted"):
        return None
    match = SSH_PATTERN.fullmatch(text)
    if match is None:
        raise MalformedLogLine("SSH entry does not match the expected format")
    fields = match.groupdict()

    import ipaddress

    try:
        ipaddress.IPv4Address(fields["ip"])
        port = int(fields["port"])
        if not 0 < port < 65536:
            raise ValueError("Port out of range")
    except ValueError as exc:
        raise MalformedLogLine("Invalid SSH address or port") from exc
    return {
        "timestamp": fields["timestamp"],
        "host": fields["host"],
        "result": fields["result"].lower(),
        "user": fields["user"],
        "invalid_user": fields["invalid"] is not None,
        "ip": fields["ip"],
        "port": port,
    }

class ParseSummary(TypedDict):
    events: list[AuthEvent]
    skipped: int
    malformed_lines: list[int]

def parse_auth_log_with_report(path: str | Path) -> ParseSummary:
    events: list[AuthEvent] = []
    skipped = 0
    malformed_lines: list[int] = []
    try:
        with Path(path).open("r", encoding="utf-8") as handle:
            for number, line in enumerate(handle, start=1):
                try:
                    event = parse_ssh_line(line)
                except MalformedLogLine as exc:
                    malformed_lines.append(number)
                    log.warning("Malformed SSH log at line %s: %s", number, exc)
                    continue
                if event is None:
                    skipped += 1
                else:
                    events.append(event)
    except FileNotFoundError as exc:
        raise AuthLogFileError(f"Authentication log not found: {path}") from exc
    except PermissionError as exc:
        raise AuthLogFileError(f"Permission denied reading auth log: {path}") from exc
    return {"events": events, "skipped": skipped, "malformed_lines": malformed_lines}

def parse_auth_log(path: str | Path) -> list[AuthEvent]:
    return parse_auth_log_with_report(path)["events"]


### C1.2 — Обнаружение brute-force атак

In [ ]:
%%writefile seclab/c1/brute_force.py
from __future__ import annotations

from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from typing import Iterable, TypedDict

from .auth import AuthEvent

class BruteForceAlert(TypedDict):
    ip: str
    attempt_count: int
    first_timestamp: str
    last_timestamp: str
    targeted_usernames: list[str]

def timestamped_events(
    events: Iterable[AuthEvent], year: int, source_timezone: str = "UTC"
) -> list[tuple[AuthEvent, datetime]]:
    tz = ZoneInfo(source_timezone)
    current_year = year
    previous_local: datetime | None = None
    converted: list[tuple[AuthEvent, datetime]] = []
    for event in events:
        local_naive = datetime.strptime(
            f"{current_year} {event['timestamp']}", "%Y %b %d %H:%M:%S"
        )
        if (
            previous_local is not None
            and previous_local.month == 12
            and local_naive.month == 1
            and (previous_local - local_naive).days > 300
        ):
            current_year += 1
            local_naive = local_naive.replace(year=current_year)
        previous_local = local_naive
        converted.append((event, local_naive.replace(tzinfo=tz).astimezone(timezone.utc)))
    return converted

def detect_brute_force(
    events: Iterable[AuthEvent],
    year: int,
    source_timezone: str = "UTC",
    threshold: int = 5,
    window_seconds: int = 60,
) -> list[BruteForceAlert]:
    if threshold < 1 or window_seconds <= 0:
        raise ValueError("threshold must be positive and window_seconds > 0")
    failed = [(e, dt) for e, dt in timestamped_events(events, year, source_timezone)
              if e["result"] == "failed"]
    failed.sort(key=lambda item: item[1])
    queues: dict[str, deque[tuple[AuthEvent, datetime]]] = defaultdict(deque)
    active: dict[str, bool] = defaultdict(bool)
    alerts: list[BruteForceAlert] = []
    for event, when in failed:
        ip = event["ip"]
        queue = queues[ip]
        while queue and when - queue[0][1] > timedelta(seconds=window_seconds):
            queue.popleft()
        if len(queue) < threshold:
            active[ip] = False
        queue.append((event, when))
        if len(queue) >= threshold and not active[ip]:
            alerts.append({
                "ip": ip,
                "attempt_count": len(queue),
                "first_timestamp": queue[0][1].isoformat(),
                "last_timestamp": queue[-1][1].isoformat(),
                "targeted_usernames": sorted({ev["user"] for ev, _ in queue}),
            })
            active[ip] = True
    return alerts


### C1.3 — Извлечение индикаторов компрометации (IOC)

In [ ]:
%%writefile seclab/c1/ioc.py
from __future__ import annotations

import ipaddress
import json
import re
from pathlib import Path

HASH_PATTERN = re.compile(r"\b(?:[a-fA-F0-9]{64}|[a-fA-F0-9]{40}|[a-fA-F0-9]{32})\b")
URL_PATTERN = re.compile(r"\bhttps?://[A-Za-z0-9._~%!$&'()*+,;=:@/?#\[\]-]+", re.I)
EMAIL_PATTERN = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
IP_PATTERN = re.compile(r"(?<![\w.])\d{1,3}(?:\.\d{1,3}){3}(?![\w.])")
DOMAIN_PATTERN = re.compile(
    r"(?<![\w@.-])(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+"
    r"[A-Za-z]{2,63}(?![\w.-])"
)

def refang(text: str) -> str:
    text = re.sub(r"hxxps://", "https://", text, flags=re.I)
    text = re.sub(r"hxxp://", "http://", text, flags=re.I)
    return text.replace("[.]", ".").replace("(.)", ".").replace("[@]", "@")

def _clear_spans(text: str, matches: list[re.Match[str]]) -> str:
    chars = list(text)
    for match in matches:
        chars[match.start():match.end()] = " " * (match.end() - match.start())
    return "".join(chars)

def extract_iocs(text: str) -> dict[str, list[str]]:
    cleaned = refang(text)
    hashes = HASH_PATTERN.findall(cleaned)
    no_hashes = _clear_spans(cleaned, list(HASH_PATTERN.finditer(cleaned)))
    urls = [m.group().rstrip(".,;:!?)\"'") for m in URL_PATTERN.finditer(no_hashes)]
    no_urls = _clear_spans(no_hashes, list(URL_PATTERN.finditer(no_hashes)))
    email_matches = list(EMAIL_PATTERN.finditer(no_urls))
    emails = [m.group() for m in email_matches]
    no_emails = _clear_spans(no_urls, email_matches)
    ips = []
    ip_matches = list(IP_PATTERN.finditer(no_emails))
    for match in ip_matches:
        candidate = match.group()
        try:
            ips.append(str(ipaddress.IPv4Address(candidate)))
        except ValueError:
            pass
    no_ips = _clear_spans(no_emails, ip_matches)
    domains = [m.group().lower() for m in DOMAIN_PATTERN.finditer(no_ips)]

    return {
        "ipv4": sorted(set(ips)),
        "domains": sorted(set(domains)),
        "urls": sorted(set(urls)),
        "emails": sorted(set(emails)),
        "md5": sorted({h.lower() for h in hashes if len(h) == 32}),
        "sha1": sorted({h.lower() for h in hashes if len(h) == 40}),
        "sha256": sorted({h.lower() for h in hashes if len(h) == 64}),
    }

def extract_iocs_from_file(report_path: str | Path, output_path: str | Path = "iocs.json") -> dict[str, list[str]]:
    results = extract_iocs(Path(report_path).read_text(encoding="utf-8"))
    Path(output_path).write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return results


### C1.4 — Проверка политики паролей

In [ ]:
%%writefile seclab/c1/password_audit.py
from __future__ import annotations

import json
import re
from pathlib import Path

REPEAT_PATTERN = re.compile(r"(.)\1{2,}")
SYMBOL_PATTERN = re.compile(r"[^A-Za-z0-9]")

def audit_passwords(passwords_file: str | Path, common_file: str | Path, output_file: str | Path | None = None) -> list[dict[str, object]]:
    common = {
        item.strip().casefold()
        for item in Path(common_file).read_text(encoding="utf-8").splitlines()
        if item.strip()
    }
    report: list[dict[str, object]] = []
    with Path(passwords_file).open("r", encoding="utf-8") as handle:
        for line_number, password_line in enumerate(handle, start=1):
            password = password_line.rstrip("\r\n")
            broken: list[str] = []
            if len(password) < 12:
                broken.append("minimum_length_12")
            if re.search(r"[A-Z]", password) is None:
                broken.append("missing_uppercase")
            if re.search(r"[a-z]", password) is None:
                broken.append("missing_lowercase")
            if re.search(r"[0-9]", password) is None:
                broken.append("missing_digit")
            if SYMBOL_PATTERN.search(password) is None:
                broken.append("missing_symbol")
            if REPEAT_PATTERN.search(password) is not None:
                broken.append("repeated_character_3plus")
            if password.casefold() in common:
                broken.append("common_password")
            report.append({"line": line_number, "broken_rules": broken})
    if output_file is not None:
        Path(output_file).write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report


### C1.5 — Контроль целостности файлов

In [ ]:
%%writefile seclab/c1/integrity.py
from __future__ import annotations

import argparse
import hashlib
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

CHUNK_SIZE = 64 * 1024

def _metadata(path: Path) -> dict[str, Any]:
    sha = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_SIZE), b""):
            sha.update(chunk)
        stat = path.stat()
    return {
        "sha256": sha.hexdigest(),
        "size": stat.st_size,
        "mtime_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
    }

def _scan(directory: Path, excluded: Path | None = None) -> tuple[dict[str, dict[str, Any]], list[str]]:
    directory = directory.resolve()
    excluded = excluded.resolve() if excluded is not None else None
    files: dict[str, dict[str, Any]] = {}
    unreadable: list[str] = []
    for root, dirs, filenames in os.walk(directory):
        dirs.sort()
        for filename in sorted(filenames):
            path = Path(root) / filename
            if excluded is not None and path.resolve() == excluded:
                continue
            relative = path.relative_to(directory).as_posix()
            try:
                files[relative] = _metadata(path)
            except (PermissionError, FileNotFoundError, IsADirectoryError, OSError):
                unreadable.append(relative)
    return files, unreadable

def create_baseline(directory: str | Path, baseline_path: str | Path = "baseline.json") -> dict[str, Any]:
    root = Path(directory)
    if not root.is_dir():
        raise NotADirectoryError(f"Directory not found: {root}")
    destination = Path(baseline_path).resolve()
    files, unreadable = _scan(root, destination)
    baseline: dict[str, Any] = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "files": files,
        "unreadable": unreadable,
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary: Path | None = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=destination.parent, prefix=".baseline-", suffix=".tmp", delete=False) as file:
            temporary = Path(file.name)
            json.dump(baseline, file, indent=2, ensure_ascii=False)
            file.write("\n")
            file.flush()
            os.fsync(file.fileno())
        temporary.replace(destination)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return baseline

def check_integrity(directory: str | Path, baseline_path: str | Path = "baseline.json") -> dict[str, list[str]]:
    baseline_file = Path(baseline_path).resolve()
    baseline = json.loads(baseline_file.read_text(encoding="utf-8"))
    previous: dict[str, dict[str, Any]] = baseline["files"]
    current, unreadable = _scan(Path(directory), baseline_file)
    previous_paths = set(previous)
    current_paths = set(current)
    unknown = set(unreadable)

    return {
        "added": sorted(current_paths - previous_paths),
        "removed": sorted(previous_paths - current_paths - unknown),
        "modified": sorted(path for path in current_paths & previous_paths
                           if current[path] != previous[path]),
        "unreadable": sorted(unreadable),
    }

def main() -> None:
    parser = argparse.ArgumentParser(description="C1.5 file integrity monitor")
    parser.add_argument("command", choices=["baseline", "check"])
    parser.add_argument("directory")
    parser.add_argument("--baseline", default="baseline.json")
    args = parser.parse_args()
    if args.command == "baseline":
        result = create_baseline(args.directory, args.baseline)
    else:
        result = check_integrity(args.directory, args.baseline)
    print(json.dumps(result, indent=2))

if __name__ == "__main__":
    main()


### C1.6 — Проверка срока действия TLS-сертификатов

In [ ]:
%%writefile seclab/c1/tls.py
from __future__ import annotations

import csv
import math
from datetime import datetime, timezone
from pathlib import Path

class UnknownDateFormat(ValueError):
    pass

DATE_FORMATS = ("%b %d %H:%M:%S %Y GMT", "%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S%z")

def parse_not_after(value: str) -> datetime:
    for format_string in DATE_FORMATS:
        try:
            dt = datetime.strptime(value, format_string)
            return dt.replace(tzinfo=timezone.utc) if dt.tzinfo is None else dt.astimezone(timezone.utc)
        except ValueError:
            continue
    raise UnknownDateFormat(f"Unsupported certificate expiry date format: {value!r}")

def check_certificates(inventory_csv: str | Path, output_csv: str | Path, now: datetime) -> list[dict[str, str]]:
    if now.tzinfo is None or now.utcoffset() is None:
        raise ValueError("now must be timezone-aware")
    now = now.astimezone(timezone.utc)
    result: list[dict[str, str]] = []
    with Path(inventory_csv).open(newline="", encoding="utf-8") as source:
        for row in csv.DictReader(source):
            expires = parse_not_after(row["notAfter"])
            seconds = (expires - now).total_seconds()
            days = math.floor(seconds / 86400)
            if seconds <= 0:
                status = "expired"
            elif seconds < 14 * 86400:
                status = "critical"
            elif seconds < 30 * 86400:
                status = "warning"
            else:
                status = "ok"
            result.append({
                "hostname": row["hostname"],
                "notAfter": expires.isoformat(),
                "days_remaining": str(days),
                "status": status,
            })
    result.sort(key=lambda r: (int(r["days_remaining"]), r["hostname"]))
    with Path(output_csv).open("w", newline="", encoding="utf-8") as target:
        writer = csv.DictWriter(target, fieldnames=["hostname", "notAfter", "days_remaining", "status"])
        writer.writeheader()
        writer.writerows(result)
    return result


## C2 — Классы, наследование и полиморфизм

### C2.1 — Модель события безопасности

In [ ]:
%%writefile seclab/c2/models.py
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from ipaddress import IPv4Address, ip_address
from typing import Any, Mapping

class Severity(Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"

class EventParseError(ValueError):
    pass

@dataclass(frozen=True, order=True)
class SecurityEvent:
    timestamp: datetime
    source_ip: IPv4Address = field(compare=False)
    user: str = field(compare=False)
    action: str = field(compare=False)
    severity: Severity = field(compare=False)

    def __post_init__(self) -> None:
        offset = self.timestamp.utcoffset()
        if self.timestamp.tzinfo is None or offset is None:
            raise ValueError("timestamp must be timezone-aware")
        if offset.total_seconds() != 0:
            raise ValueError("timestamp must be in UTC")
        if not self.action.strip():
            raise ValueError("action must not be empty")
        if not isinstance(self.source_ip, IPv4Address):
            raise ValueError("source_ip must be an IPv4Address")
        if not isinstance(self.severity, Severity):
            raise ValueError("severity must be a Severity enum")

    @classmethod
    def from_dict(cls, data: Mapping[str, Any]) -> SecurityEvent:
        try:
            raw_timestamp = data["timestamp"]
            if isinstance(raw_timestamp, datetime):
                timestamp = raw_timestamp
            else:
                timestamp = datetime.fromisoformat(str(raw_timestamp).replace("Z", "+00:00"))
            if timestamp.tzinfo is None or timestamp.utcoffset() is None:
                raise ValueError("Timestamp lacks timezone")
            timestamp = timestamp.astimezone(timezone.utc)
            address = ip_address(str(data["source_ip"]))
            if not isinstance(address, IPv4Address):
                raise ValueError("IPv6 addresses are not supported")
            raw_severity = data["severity"]
            severity = (raw_severity if isinstance(raw_severity, Severity)
                        else Severity[str(raw_severity).upper()])
            return cls(timestamp, address, str(data["user"]), str(data["action"]), severity)
        except (ValueError, KeyError, TypeError) as exc:
            raise EventParseError("Invalid security event record") from exc

@dataclass(frozen=True)
class Alert:
    rule: str
    severity: Severity
    timestamp: datetime
    source_ip: IPv4Address
    message: str

    def to_dict(self) -> dict[str, str]:
        return {
            "rule": self.rule,
            "severity": self.severity.value,
            "timestamp": self.timestamp.astimezone(timezone.utc).isoformat(),
            "source_ip": str(self.source_ip),
            "message": self.message,
        }


### C2.2 — Иерархия парсеров с ABC

In [ ]:
%%writefile seclab/c2/parsers.py
from __future__ import annotations

import json
import logging
import re
from abc import ABC, abstractmethod
from datetime import datetime, timezone
from ipaddress import IPv4Address
from pathlib import Path
from typing import Any, ClassVar
from zoneinfo import ZoneInfo

from seclab.c1.auth import MalformedLogLine, parse_ssh_line
from .models import EventParseError, SecurityEvent, Severity

log = logging.getLogger(__name__)

class LogParser(ABC):
    _registry: ClassVar[dict[str, type[LogParser]]] = {}

    def __init_subclass__(cls, *, format_name: str | None = None, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        if format_name is not None:
            if format_name in cls._registry:
                raise ValueError(f"Duplicate parser format: {format_name}")
            cls._registry[format_name] = cls

    def __init__(self) -> None:
        self.malformed_lines: list[int] = []
        self.skipped = 0

    @abstractmethod
    def parse_line(self, line: str) -> SecurityEvent | None:
        raise NotImplementedError

    def parse_file(self, path: str | Path) -> list[SecurityEvent]:
        self.malformed_lines = []
        self.skipped = 0
        result: list[SecurityEvent] = []
        with Path(path).open("r", encoding="utf-8") as source:
            for line_number, line in enumerate(source, 1):
                try:
                    event = self.parse_line(line.rstrip("\n"))
                except (ValueError, KeyError, TypeError) as exc:
                    self.malformed_lines.append(line_number)
                    log.warning("Malformed line %d: %s", line_number, type(exc).__name__)
                else:
                    if event is None:
                        self.skipped += 1
                    else:
                        result.append(event)
        return result

    @classmethod
    def for_format(cls, name: str, **kwargs: Any) -> LogParser:
        try:
            return cls._registry[name](**kwargs)
        except KeyError as exc:
            raise ValueError(f"Unknown parser: {name}") from exc

class SshAuthParser(LogParser, format_name="ssh"):
    def __init__(self, year: int = 2026, source_timezone: str = "UTC") -> None:
        super().__init__()
        self.year = year
        self.source_tz = ZoneInfo(source_timezone)
        self._previous: datetime | None = None

    def parse_line(self, line: str) -> SecurityEvent | None:
        event = parse_ssh_line(line)
        if event is None:
            return None
        dt = datetime.strptime(f"{self.year} {event['timestamp']}", "%Y %b %d %H:%M:%S")
        if (self._previous is not None and self._previous.month == 12 and dt.month == 1
                and (self._previous - dt).days > 300):
            self.year += 1
            dt = dt.replace(year=self.year)
        self._previous = dt
        utc_timestamp = dt.replace(tzinfo=self.source_tz).astimezone(timezone.utc)
        success = event["result"] == "accepted"
        return SecurityEvent(
            timestamp=utc_timestamp,
            source_ip=IPv4Address(event["ip"]),
            user=event["user"],
            action="login_success" if success else "login_failed",
            severity=Severity.LOW if success else Severity.MEDIUM,
        )

COMBINED_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ (?P<user>\S+) \[(?P<time>[^]]+)\] '
    r'"(?P<method>\S+) (?P<resource>.*?) HTTP/[\d.]+" '
    r'(?P<status>\d{3}) (?P<bytes>\d+|-) "(?:[^"\\]|\\.)*" "(?:[^"\\]|\\.)*"'
)

class WebAccessParser(LogParser, format_name="apache"):
    def parse_line(self, line: str) -> SecurityEvent | None:
        if not line.strip():
            return None
        match = COMBINED_PATTERN.fullmatch(line)
        if match is None:
            raise ValueError("Bad combined Apache log line")
        parts = match.groupdict()
        ts = datetime.strptime(parts["time"], "%d/%b/%Y:%H:%M:%S %z")
        status = int(parts["status"])
        severity = (Severity.HIGH if status >= 500 else
                    Severity.MEDIUM if status >= 400 else Severity.LOW)
        return SecurityEvent(
            timestamp=ts.astimezone(timezone.utc),
            source_ip=IPv4Address(parts["ip"]),
            user=parts["user"] if parts["user"] != "-" else "anonymous",
            action=f"http_{parts['method'].lower()}_{status}",
            severity=severity,
        )

class JsonLineParser(LogParser, format_name="jsonl"):
    def parse_line(self, line: str) -> SecurityEvent | None:
        if not line.strip():
            return None
        try:
            value = json.loads(line)
            if not isinstance(value, dict):
                raise ValueError("JSON event must be an object")
            return SecurityEvent.from_dict(value)
        except (json.JSONDecodeError, EventParseError) as exc:
            raise ValueError("Malformed JSON event") from exc


### C2.3 — Движок правил обнаружения

In [ ]:
%%writefile seclab/c2/rules.py
from __future__ import annotations

import logging
from abc import ABC, abstractmethod
from collections import defaultdict, deque
from datetime import datetime, time, timedelta
from ipaddress import IPv4Address
from typing import Iterable, Mapping, Sequence
from zoneinfo import ZoneInfo

from .models import Alert, SecurityEvent, Severity

logger = logging.getLogger(__name__)

class DetectionRule(ABC):
    name: str
    severity: Severity

    @abstractmethod
    def evaluate(self, events: Sequence[SecurityEvent]) -> list[Alert]:
        raise NotImplementedError

class BruteForceRule(DetectionRule):
    name = "brute_force"
    severity = Severity.HIGH

    def __init__(self, threshold: int = 5, window_seconds: int = 60) -> None:
        if threshold < 1 or window_seconds <= 0:
            raise ValueError("Invalid brute-force threshold/window")
        self.threshold = threshold
        self.window = timedelta(seconds=window_seconds)

    def evaluate(self, events: Sequence[SecurityEvent]) -> list[Alert]:
        queues: dict[IPv4Address, deque[SecurityEvent]] = defaultdict(deque)
        active: dict[IPv4Address, bool] = defaultdict(bool)
        alerts: list[Alert] = []
        for event in sorted(events):
            if event.action != "login_failed":
                continue
            queue = queues[event.source_ip]
            while queue and event.timestamp - queue[0].timestamp > self.window:
                queue.popleft()
            if len(queue) < self.threshold:
                active[event.source_ip] = False
            queue.append(event)
            if len(queue) >= self.threshold and not active[event.source_ip]:
                names = ", ".join(sorted({item.user for item in queue}))
                alerts.append(Alert(
                    rule=self.name, severity=self.severity, timestamp=event.timestamp,
                    source_ip=event.source_ip,
                    message=f"{len(queue)} failed logins in {self.window.total_seconds():g}s; users: {names}",
                ))
                active[event.source_ip] = True
        return alerts

class OffHoursLoginRule(DetectionRule):
    name = "off_hours_login"
    severity = Severity.MEDIUM

    def __init__(self, source_timezone: str = "UTC", start_hour: int = 8, end_hour: int = 20) -> None:
        if not (0 <= start_hour < end_hour <= 24):
            raise ValueError("Hours must satisfy 0 <= start < end <= 24")
        self.timezone = ZoneInfo(source_timezone)
        self.start_hour = start_hour
        self.end_hour = end_hour

    def evaluate(self, events: Sequence[SecurityEvent]) -> list[Alert]:
        alerts: list[Alert] = []
        for event in events:
            if event.action != "login_success":
                continue
            local_hour = event.timestamp.astimezone(self.timezone).hour
            if not self.start_hour <= local_hour < self.end_hour:
                alerts.append(Alert(
                    rule=self.name, severity=self.severity,
                    timestamp=event.timestamp, source_ip=event.source_ip,
                    message=f"Off-hours login for {event.user} ({self.timezone.key})",
                ))
        return alerts

class NewSourceForUserRule(DetectionRule):
    name = "new_source_for_user"
    severity = Severity.MEDIUM

    def __init__(self, known_sources: Mapping[str, Iterable[IPv4Address]] | None = None) -> None:
        self.known_sources = {
            user: set(ips) for user, ips in (known_sources or {}).items()
        }

    def evaluate(self, events: Sequence[SecurityEvent]) -> list[Alert]:
        known = {user: set(ips) for user, ips in self.known_sources.items()}
        alerts: list[Alert] = []
        for event in sorted(events):
            if event.action != "login_success":
                continue
            if event.user not in known:

                known[event.user] = {event.source_ip}
            elif event.source_ip not in known[event.user]:
                alerts.append(Alert(
                    rule=self.name, severity=self.severity, timestamp=event.timestamp,
                    source_ip=event.source_ip,
                    message=f"New source IP for user {event.user}",
                ))
                known[event.user].add(event.source_ip)
        return alerts

class DetectionEngine:
    def __init__(self, rules: Sequence[DetectionRule]) -> None:
        self.rules = list(rules)

    def run(self, events: Sequence[SecurityEvent]) -> list[Alert]:
        result: list[Alert] = []
        for rule in self.rules:
            try:
                result.extend(rule.evaluate(events))
            except Exception:
                logger.exception("Rule %s failed, proceeding with remaining rules", rule.name)
        return result


### C2.4 — Безопасное отображение секретов

In [ ]:
%%writefile seclab/c2/secret.py
from __future__ import annotations

from hmac import compare_digest

class Secret:
    __slots__ = ("__value",)

    def __init__(self, value: str) -> None:
        if not isinstance(value, str):
            raise TypeError("Secret requires a string")
        self.__value = value

    def reveal(self) -> str:
        return self.__value

    def __repr__(self) -> str:
        return "Secret('********')"

    def __str__(self) -> str:
        return self.__repr__()

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Secret):
            return NotImplemented
        return compare_digest(self.__value, other.__value)

    def __hash__(self) -> int:

        return hash(self.__value)


### C2.5 — Исправление нарушений LSP

In [ ]:
%%writefile seclab/c2/notifiers.py
from __future__ import annotations

from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> bool:
        raise NotImplementedError

class EmailNotifier(Notifier):
    def __init__(self, reachable: bool = True) -> None:
        self.reachable = reachable
        self.sent: list[str] = []

    def send(self, message: str) -> bool:
        if not self.reachable:
            return False
        self.sent.append(message)
        return True

class SmsNotifier(Notifier):
    def __init__(self, reachable: bool = True) -> None:
        self.reachable = reachable
        self.sent_chunks: list[str] = []

    def send(self, message: str) -> bool:
        if not self.reachable:
            return False

        chunks = [message[i:i + 160] for i in range(0, len(message), 160)] or [""]
        self.sent_chunks.extend(chunks)
        return True

class WebhookNotifier(Notifier):
    def __init__(self, reachable: bool = True) -> None:
        self.reachable = reachable
        self.sent: list[str] = []

    def send(self, message: str) -> bool:
        if not self.reachable:
            return False
        self.sent.append(message)
        return True


### C2.6 — Способы вывода оповещений через Protocol

In [ ]:
%%writefile seclab/c2/sinks.py
from __future__ import annotations

import csv
import json
import logging
import sys
from pathlib import Path
from typing import Protocol, Sequence, TextIO

from .models import Alert

logger = logging.getLogger(__name__)

class AlertSink(Protocol):
    def send(self, alert: Alert) -> None: ...

class ConsoleSink:
    def __init__(self, stream: TextIO | None = None) -> None:
        self.stream = stream

    def send(self, alert: Alert) -> None:
        destination = self.stream if self.stream is not None else sys.stdout
        print(f"[{alert.severity.value}] {alert.timestamp.isoformat()} {alert.rule}: {alert.message}", file=destination)

class JsonLinesSink:
    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)

    def send(self, alert: Alert) -> None:
        with self.path.open("a", encoding="utf-8") as destination:
            destination.write(json.dumps(alert.to_dict()) + "\n")

class CsvSink:
    columns = ("rule", "severity", "timestamp", "source_ip", "message")

    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)

    def send(self, alert: Alert) -> None:
        need_header = not self.path.exists() or self.path.stat().st_size == 0
        with self.path.open("a", newline="", encoding="utf-8") as destination:
            writer = csv.DictWriter(destination, fieldnames=self.columns)
            if need_header:
                writer.writeheader()
            writer.writerow(alert.to_dict())

class MultiSink:
    def __init__(self, sinks: Sequence[AlertSink]) -> None:
        self.sinks = list(sinks)

    def send(self, alert: Alert) -> None:
        for sink in self.sinks:
            try:
                sink.send(alert)
            except Exception:
                logger.exception("Failed to send alert to sink %s", type(sink).__name__)


## Учебные данные и пример запуска

In [ ]:
from pathlib import Path
fixtures = {
    'examples/data/auth.log': 'Oct 19 08:15:02 srv01 sshd[2211]: Failed password for invalid user admin from 203.0.113.45 port 52144 ssh2\nOct 19 08:15:04 srv01 sshd[2211]: Failed password for root from 203.0.113.45 port 52150 ssh2\nOct 19 08:16:40 srv01 sshd[2290]: Accepted publickey for aru from 198.51.100.7 port 40112 ssh2\nOct 19 08:17:01 srv01 CRON[2301]: pam_unix(cron:session): session opened for user root\n### corrupted line ###\n',
    'examples/data/common_passwords.txt': 'password123\nqwerty\n123456\n',
    'examples/data/passwords.txt': 'short\nAAAAAAAAAAAA1!\nNoDigitHere!!\nGoodPassword12!\npassword123\n',
    'examples/data/threat_report.txt': 'Threat intelligence report (synthetic). The 12 VALID stand-alone indicators are:\nIPv4 203[.]0[.]113[.]45\nIPv4 198.51.100.7\nDomain malicious[.]example\nDomain c2.example.org\nURL hxxps://bad[.]example/collect\nURL hxxp://evil[.]example/payload\nEmail admin[@]bad.example\nEmail alerts@example.org\nMD5 d41d8cd98f00b204e9800998ecf8427e\nSHA1 da39a3ee5e6b4b0d3255bfef95601890afd80709\nSHA256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\nDomain compromised.net\n\n5 DECOYS (must not be detected):\nInvalid IPv4 999.999.999.999\nInvalid partial IP 1.1.1\nInvalid hash extended 0d41d8cd98f00b204e9800998ecf8427e0\nInvalid domain a..b\nInvalid email user@localhost\n',
    'examples/data/tls_inventory.csv': 'hostname,notAfter\nexpired.example,Oct 19 12:00:00 2026 GMT\ncritical.example,2026-11-10T23:59:59Z\nwarning.example,2026-11-25T23:59:59Z\nok.example,2026-12-30T00:00:00Z\n',
}
for filename, content in fixtures.items():
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    Path(filename).write_text(content, encoding="utf-8")
print("Created", len(fixtures), "example files")


In [ ]:
%%writefile examples/demo.py
from __future__ import annotations

import json
import tempfile
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Address
from pathlib import Path

from seclab.c1.auth import parse_auth_log_with_report
from seclab.c1.brute_force import detect_brute_force
from seclab.c1.ioc import extract_iocs_from_file
from seclab.c1.password_audit import audit_passwords
from seclab.c1.integrity import create_baseline, check_integrity
from seclab.c1.tls import check_certificates
from seclab.c2.models import Alert, SecurityEvent, Severity
from seclab.c2.parsers import LogParser
from seclab.c2.rules import DetectionEngine, BruteForceRule, OffHoursLoginRule, NewSourceForUserRule
from seclab.c2.secret import Secret
from seclab.c2.notifiers import EmailNotifier, SmsNotifier, WebhookNotifier
from seclab.c2.sinks import ConsoleSink, CsvSink, JsonLinesSink, MultiSink

DATA = Path(__file__).resolve().parent / "data"

def run_demo() -> None:
    print("C1.1 SSH log:")
    summary = parse_auth_log_with_report(DATA / "auth.log")
    print(json.dumps(summary, indent=2))

    print("\nC1.2 Brute force with 5 synthetic events:")
    from seclab.c1.auth import AuthEvent
    failures: list[AuthEvent] = [
        {"timestamp": f"Nov 08 12:00:{i:02d}", "host": "demo", "result": "failed",
         "user": "admin", "invalid_user": True, "ip": "203.0.113.45", "port": 22}
        for i in range(5)
    ]
    print(detect_brute_force(failures, 2026, "Asia/Almaty"))

    with tempfile.TemporaryDirectory() as folder:
        out = Path(folder)
        print("\nC1.3 IOC counts:")
        iocs = extract_iocs_from_file(DATA / "threat_report.txt", out / "iocs.json")
        print({type_: len(values) for type_, values in iocs.items()})
        print("\nC1.4 Password audit (no password contents):")
        print(audit_passwords(DATA / "passwords.txt", DATA / "common_passwords.txt"))
        print("\nC1.5 File integrity:")
        example_dir = out / "files"
        example_dir.mkdir()
        sample = example_dir / "a.txt"
        sample.write_text("before")
        create_baseline(example_dir, out / "baseline.json")
        sample.write_text("after")
        (example_dir / "new.txt").write_text("new")
        print(check_integrity(example_dir, out / "baseline.json"))
        print("\nC1.6 TLS expiration:")
        print(check_certificates(DATA / "tls_inventory.csv", out / "certs.csv", datetime(2026, 11, 8, tzinfo=timezone.utc)))

        print("\nC2.1 SecurityEvent:")
        t0 = datetime(2026, 11, 8, 2, 0, tzinfo=timezone.utc)
        event = SecurityEvent(t0, IPv4Address("203.0.113.45"), "aru", "login_success", Severity.LOW)
        print(event)
        print("\nC2.2 LogParser registry:")
        parser = LogParser.for_format("ssh", year=2026, source_timezone="Asia/Almaty")
        print("events:", len(parser.parse_file(DATA / "auth.log")), "; malformed:", parser.malformed_lines)
        print("\nC2.3 Rules:")
        batch = [SecurityEvent(t0 + timedelta(seconds=i * 10), event.source_ip, "aru", "login_failed", Severity.MEDIUM) for i in range(5)]
        batch.append(event)
        print([a.to_dict() for a in DetectionEngine([BruteForceRule(), OffHoursLoginRule("UTC"), NewSourceForUserRule()]).run(batch)])
        print("\nC2.4 Secret redaction:")
        secret = Secret("hidden_api_token")
        print(secret, repr({secret: 1}))
        print("\nC2.5 Notifier LSP contract:")
        print({cls.__name__: cls().send("x" * 350) for cls in (EmailNotifier, SmsNotifier, WebhookNotifier)})
        print("\nC2.6 MultiSink (writes JSONL and CSV):")
        alert = Alert("test", Severity.HIGH, t0, event.source_ip, "Example alert")
        MultiSink([ConsoleSink(), JsonLinesSink(out / "alerts.jsonl"), CsvSink(out / "alerts.csv")]).send(alert)
        print("JSONL:", (out / "alerts.jsonl").read_text().strip())

if __name__ == "__main__":
    run_demo()


## Демонстрация всех заданий

In [ ]:
!python -m examples.demo

## Автоматические тесты

In [ ]:
%%writefile tests/test_c1.py
from __future__ import annotations

import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import pytest

from seclab.c1.auth import AuthLogFileError, parse_auth_log, parse_auth_log_with_report
from seclab.c1.brute_force import detect_brute_force, timestamped_events
from seclab.c1.ioc import extract_iocs, extract_iocs_from_file
from seclab.c1.password_audit import audit_passwords
from seclab.c1.integrity import create_baseline, check_integrity
from seclab.c1.tls import UnknownDateFormat, parse_not_after, check_certificates

DATA = Path(__file__).resolve().parent.parent / "examples" / "data"

def _failed(times: list[datetime], ip: str = "203.0.113.45") -> list[dict]:
    return [{
        "timestamp": time.strftime("%b %d %H:%M:%S"),
        "host": "srv01", "result": "failed", "user": f"u{i}",
        "invalid_user": False, "ip": ip, "port": 22,
    } for i, time in enumerate(times)]

def test_c11_sample_and_missing_file(tmp_path: Path) -> None:
    report = parse_auth_log_with_report(DATA / "auth.log")
    assert len(report["events"]) == 3
    assert report["skipped"] == 1
    assert report["malformed_lines"] == [5]
    assert report["events"][0]["invalid_user"]
    assert report["events"][1]["result"] == "failed"
    assert len(parse_auth_log(DATA / "auth.log")) == 3
    with pytest.raises(AuthLogFileError, match="not found"):
        parse_auth_log(tmp_path / "missing.log")

def test_c11_malformed_sshd(tmp_path: Path) -> None:
    f = tmp_path / "auth.log"
    f.write_text("Oct 19 sshd broken entry\n", encoding="utf-8")
    assert parse_auth_log_with_report(f)["malformed_lines"] == [1]

def test_c12_four_fails_no_alert() -> None:
    start = datetime(2026, 11, 8, 12, 0, 0)
    assert detect_brute_force(_failed([start + timedelta(seconds=i * 10) for i in range(4)]), 2026) == []

def test_c12_five_fails_in_59_seconds() -> None:
    start = datetime(2026, 11, 8, 12, 0, 0)
    events = _failed([start + timedelta(seconds=i) for i in [0, 12, 24, 48, 59]])
    alert, = detect_brute_force(events, 2026)
    assert alert["attempt_count"] == 5
    assert alert["targeted_usernames"] == ["u0", "u1", "u2", "u3", "u4"]
    assert alert["first_timestamp"].endswith("+00:00")

def test_c12_five_fails_in_61_seconds_no_alert() -> None:
    start = datetime(2026, 11, 8, 12, 0, 0)
    events = _failed([start + timedelta(seconds=i) for i in [0, 15, 30, 45, 61]])
    assert detect_brute_force(events, 2026) == []

def test_c12_year_rollover_and_source_timezone() -> None:
    times = [datetime(2026, 12, 31, 23, 59, 58) + timedelta(seconds=i)
             for i in [0, 1, 2, 4, 5]]
    events = _failed(times)
    converted = timestamped_events(events, 2026, "Asia/Almaty")
    assert converted[0][1] == datetime(2026, 12, 31, 18, 59, 58, tzinfo=timezone.utc)
    assert converted[-1][1].year == 2026
    assert len(detect_brute_force(events, 2026, "Asia/Almaty")) == 1

def test_c13_exact_twelve_and_no_decoys(tmp_path: Path) -> None:
    result = extract_iocs_from_file(DATA / "threat_report.txt", tmp_path / "iocs.json")
    assert sum(map(len, result.values())) == 12
    assert result["ipv4"] == ["198.51.100.7", "203.0.113.45"]
    assert len(result["domains"]) == 3
    assert len(result["urls"]) == 2
    assert len(result["emails"]) == 2
    assert [len(result[k]) for k in ["md5", "sha1", "sha256"]] == [1, 1, 1]
    assert json.loads((tmp_path / "iocs.json").read_text()) == result
    assert "1.1.1" not in str(result)
    assert not extract_iocs("f" * 33)["md5"]

def test_c14_no_password_leak(tmp_path: Path, capsys: pytest.CaptureFixture[str]) -> None:
    target = tmp_path / "audit.json"
    result = audit_passwords(DATA / "passwords.txt", DATA / "common_passwords.txt", target)
    assert len(result) == 5
    assert result[3]["broken_rules"] == []
    assert "common_password" in result[4]["broken_rules"]
    exposed = str(result) + target.read_text() + capsys.readouterr().out
    for pw in (DATA / "passwords.txt").read_text().splitlines():
        assert pw not in exposed

def test_c15_files_added_removed_modified(tmp_path: Path) -> None:
    folder = tmp_path / "folder"
    folder.mkdir()
    (folder / "removed.txt").write_text("before")
    (folder / "changed.txt").write_text("old")
    (folder / "unchanged.txt").write_text("same")
    baseline = folder / "baseline.json"
    saved = create_baseline(folder, baseline)
    assert len(saved["files"]) == 3
    (folder / "removed.txt").unlink()
    (folder / "changed.txt").write_text("new contents")
    (folder / "added.txt").write_text("addition")
    report = check_integrity(folder, baseline)
    assert report == {"added": ["added.txt"], "removed": ["removed.txt"],
                      "modified": ["changed.txt"], "unreadable": []}

def test_c15_unreadable_is_not_removed(tmp_path: Path, monkeypatch: pytest.MonkeyPatch) -> None:
    import seclab.c1.integrity as integrity
    directory = tmp_path / "data"
    directory.mkdir()
    path = directory / "file.txt"
    path.write_text("initial")
    base = tmp_path / "baseline.json"
    create_baseline(directory, base)
    original = integrity._metadata

    def bad_one(p: Path):
        if p == path:
            raise PermissionError("blocked")
        return original(p)

    monkeypatch.setattr(integrity, "_metadata", bad_one)
    result = check_integrity(directory, base)
    assert result["unreadable"] == ["file.txt"]
    assert result["removed"] == []

def test_c16_sorted_statuses(tmp_path: Path) -> None:
    now = datetime(2026, 11, 8, tzinfo=timezone.utc)
    result = check_certificates(DATA / "tls_inventory.csv", tmp_path / "results.csv", now)
    assert [item["status"] for item in result] == ["expired", "critical", "warning", "ok"]
    assert [item["hostname"] for item in result] == [
        "expired.example", "critical.example", "warning.example", "ok.example"
    ]
    assert (tmp_path / "results.csv").exists()
    with pytest.raises(UnknownDateFormat):
        parse_not_after("2026/13/80")


In [ ]:
%%writefile tests/test_c2.py
from __future__ import annotations

import csv
import io
import json
import logging
from datetime import datetime, timedelta, timezone
from ipaddress import IPv4Address
from pathlib import Path

import pytest

from seclab.c2.models import Alert, EventParseError, SecurityEvent, Severity
from seclab.c2.parsers import LogParser, SshAuthParser, WebAccessParser, JsonLineParser
from seclab.c2.rules import (BruteForceRule, DetectionEngine, DetectionRule,
                             NewSourceForUserRule, OffHoursLoginRule)
from seclab.c2.secret import Secret
from seclab.c2.notifiers import Notifier, EmailNotifier, SmsNotifier, WebhookNotifier
from seclab.c2.sinks import AlertSink, ConsoleSink, CsvSink, JsonLinesSink, MultiSink

DATA = Path(__file__).resolve().parent.parent / "examples" / "data"
T0 = datetime(2026, 11, 8, 3, 0, 0, tzinfo=timezone.utc)

def _event(sec: int = 0, ip: str = "203.0.113.45", user: str = "aru", action: str = "login_failed") -> SecurityEvent:
    return SecurityEvent(T0 + timedelta(seconds=sec), IPv4Address(ip), user, action, Severity.LOW)

def test_c21_immutable_sorted_hashable_from_dict() -> None:
    events = [_event(20), _event(10), _event(30)]
    assert sorted(events) == [events[1], events[0], events[2]]
    assert len(set([_event(20), _event(20)])) == 1
    converted = SecurityEvent.from_dict({
        "timestamp": "2026-11-08T08:00:00+05:00", "source_ip": "198.51.100.1",
        "user": "a", "action": "login_success", "severity": "high",
    })
    assert converted.timestamp == T0
    assert converted.severity is Severity.HIGH
    with pytest.raises(ValueError):
        SecurityEvent(datetime(2026, 1, 1), IPv4Address("1.2.3.4"), "a", "ok", Severity.LOW)
    with pytest.raises(EventParseError) as exc_info:
        SecurityEvent.from_dict({"timestamp": "nonsense"})
    assert exc_info.value.__cause__ is not None

def test_c22_abc_registry_and_ssh() -> None:
    with pytest.raises(TypeError):
        LogParser()
    parser = LogParser.for_format("ssh", year=2026, source_timezone="Asia/Almaty")
    assert isinstance(parser, SshAuthParser)
    events = parser.parse_file(DATA / "auth.log")
    assert len(events) == 3
    assert parser.skipped == 1
    assert parser.malformed_lines == [5]
    assert events[0].timestamp.hour == 3

def test_c22_apache_jsonl_and_dynamic_parser(tmp_path: Path) -> None:
    apache = LogParser.for_format("apache")
    event = apache.parse_line('198.51.100.7 - frank [10/Oct/2026:13:55:36 +0000] "GET /index.html HTTP/1.1" 200 2326 "-" "Mozilla/5.0"')
    assert isinstance(event, SecurityEvent)
    assert event.action == "http_get_200"
    parser = LogParser.for_format("jsonl")
    f = tmp_path / "events.jsonl"
    f.write_text(json.dumps({"timestamp": T0.isoformat(), "source_ip": "203.0.113.45",
                            "user": "aru", "action": "login_success", "severity": "LOW"}) + "\nwrong-json\n")
    assert len(parser.parse_file(f)) == 1
    assert parser.malformed_lines == [2]

    class CustomParser(LogParser, format_name="custom_for_test"):
        def parse_line(self, line: str) -> SecurityEvent | None:
            return _event() if line == "valid" else None

    assert isinstance(LogParser.for_format("custom_for_test"), CustomParser)

def test_c23_three_rules_and_engine_exception_isolation(caplog: pytest.LogCaptureFixture) -> None:
    class BrokenRule(DetectionRule):
        name = "broken"
        severity = Severity.CRITICAL

        def evaluate(self, events: list[SecurityEvent]) -> list[Alert]:
            raise RuntimeError("test error")

    failures = [_event(i * 10) for i in range(5)]
    successes = [_event(61, ip="198.51.100.2", action="login_success"),
                 _event(62, ip="203.0.113.4", action="login_success")]
    rules = [BrokenRule(), BruteForceRule(), OffHoursLoginRule("UTC"), NewSourceForUserRule()]
    with caplog.at_level(logging.ERROR):
        alerts = DetectionEngine(rules).run(failures + successes)
    assert any(a.rule == "brute_force" for a in alerts)
    assert sum(a.rule == "off_hours_login" for a in alerts) == 2
    assert any(a.rule == "new_source_for_user" for a in alerts)
    assert "Rule broken failed" in caplog.text

def test_c23_rules_empty_early_4_and_late_5() -> None:
    assert BruteForceRule().evaluate([_event(i * 10) for i in range(4)]) == []
    assert BruteForceRule().evaluate([_event(i) for i in [0, 15, 30, 45, 61]]) == []

def test_c24_masking_collections_and_logging(caplog: pytest.LogCaptureFixture) -> None:
    raw = "SuperSecretApiKey_99!"
    sec = Secret(raw)
    assert sec.reveal() == raw
    assert str(sec) == "Secret('********')"
    assert repr(sec) == str(sec)
    assert f"{sec}" == str(sec)
    assert sec == Secret(raw)
    assert sec != Secret("not-the-same")
    assert hash(sec) == hash(Secret(raw))
    with caplog.at_level(logging.ERROR):
        logging.error("Token: %s", sec)
    for view in [str(sec), repr({sec: 1}), str([sec]), str(Exception(sec)), caplog.text]:
        assert raw not in view

@pytest.mark.parametrize("factory", [EmailNotifier, SmsNotifier, WebhookNotifier])
@pytest.mark.parametrize("message", ["hello", "x" * 350, ""])
def test_c25_contract_success(factory: type[Notifier], message: str) -> None:
    notifier = factory(reachable=True)
    result = notifier.send(message)
    assert result is True
    assert isinstance(result, bool)

@pytest.mark.parametrize("factory", [EmailNotifier, SmsNotifier, WebhookNotifier])
@pytest.mark.parametrize("message", ["hello", "x" * 350])
def test_c25_contract_failure(factory: type[Notifier], message: str) -> None:
    notifier = factory(reachable=False)
    result = notifier.send(message)
    assert result is False
    assert isinstance(result, bool)

def test_c25_sms_split_no_data_loss() -> None:
    notifier = SmsNotifier()
    message = "x" * 350
    assert notifier.send(message)
    assert [len(chunk) for chunk in notifier.sent_chunks] == [160, 160, 30]
    assert "".join(notifier.sent_chunks) == message

def test_c26_sinks_and_bad_sink_isolation(tmp_path: Path, caplog: pytest.LogCaptureFixture) -> None:
    alert = Alert("brute_force", Severity.HIGH, T0, IPv4Address("203.0.113.45"), "Five failures")
    stream = io.StringIO()
    jsonpath = tmp_path / "alerts.jsonl"
    csvpath = tmp_path / "alerts.csv"

    class BrokenSink:
        def send(self, alert: Alert) -> None:
            raise OSError("sink unavailable")

    all_sinks: list[AlertSink] = [ConsoleSink(stream), BrokenSink(), JsonLinesSink(jsonpath), CsvSink(csvpath)]
    with caplog.at_level(logging.ERROR):
        MultiSink(all_sinks).send(alert)
    assert "Five failures" in stream.getvalue()
    assert json.loads(jsonpath.read_text().strip())["source_ip"] == "203.0.113.45"
    with csvpath.open(newline="") as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == 1
    assert rows[0]["rule"] == "brute_force"
    assert "Failed to send alert" in caplog.text


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("pytest") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest"])
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=False)
if result.returncode != 0:
    raise AssertionError("Some automated tests failed")
print("All assignment tests passed.")


## Необязательная проверка типов (`mypy --strict`)

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("mypy") is None:
    print("mypy is not installed. In Colab run: %pip install mypy")
    print("Then re-run this cell to check strict type annotations.")
else:
    completed = subprocess.run([sys.executable, "-m", "mypy", "--strict", "seclab"], check=False)
    if completed.returncode:
        print("mypy reported type issues; inspect the messages above.")
    else:
        print("mypy --strict completed successfully.")


## Как использовать ноутбук

1. Загрузите файл `.ipynb` в Google Colab.
2. Выполните ячейки сверху вниз или выберите **Среда выполнения → Выполнить все**.
3. В конце проверьте результаты демонстрации и автоматических тестов.

Все необходимые Python-файлы и учебные данные создаются внутри Colab. Проверка `mypy` необязательна.